# Setting up the System

## To run in Google Colab

In [2]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [3]:
import openai
openai.api_key = OPENAI_API_KEY

# Med QA Casestudy

Link to the dataset:https://huggingface.co/datasets/bigbio/med_qa

In [5]:
ls

phrases_no_exclude_train.jsonl  sample_data/


In [8]:
import json

def read_jsonl(file_path):
    """
    Reads a .jsonl file line by line and returns a list of dictionaries.
    """
    data_list = []
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                # Strip whitespace/newlines and skip empty lines
                line = line.strip()
                if line:
                    data_list.append(json.loads(line))
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")

    return data_list

# Usage
file_data = read_jsonl('phrases_no_exclude_train.jsonl')

In [26]:
x=0
# Print the resulting list of dictionaries (optional)
for item in file_data:
  if (x < 5):
    print(item)
  x = x+1

{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?', 'answer': 'Nitrofurantoin', 'options': {'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Doxycycline', 'D': 'Nitrofurantoin'}, 'meta_info': 'step2&3', 'answer_idx': 'D', 'metamap_phrases': ['23 year old pregnant woman', 'weeks presents', 'burning', 'urination', 'states', 'started 1 day', 'worsening', 'drinking', 'water', 'taking cranberry extract', 'feels well', 'followed by', 'doctor', 'pregnancy', '

In [27]:
def get_response(messages,model="gpt-4o-mini"):

    response = openai.chat.completions.create(
        model=model,
        messages=messages)

    return response.choices[0].message.content

In [28]:
first_question= f'''{file_data[0]['question']}, {file_data[0]['options']}'''
first_question_answer = file_data[0]['answer']

In [29]:
second_question= f'''{file_data[1]['question']}, {file_data[1]['options']}'''
second_question_answer = file_data[1]['answer']

In [30]:
third_question= f'''{file_data[2]['question']}, {file_data[2]['options']}'''
third_question_answer = file_data[2]['answer']

In [31]:
fourth_question= f'''{file_data[3]['question']}, {file_data[3]['options']}'''
fourth_question_answer = file_data[3]['answer']

In [32]:
fifth_question= f'''{file_data[4]['question']}, {file_data[4]['options']}'''
fifth_question_answer = file_data[4]['answer']

Define the System role

In [33]:
system_role = "You are a helpful assistant that answers multiple choice questions about medical knowledge."

# Few Shot Prompting

In [34]:
user_prompt = f'''Question:{first_question}
Answer:{first_question_answer}

Question:{second_question}
Answer:{second_question_answer}

Question:{third_question}
Answer:{third_question_answer}

Question:{fourth_question}
Answer:{fourth_question_answer}

Question:{fifth_question}
Answer:
'''

In [35]:
messages=[{'role':'system','content':system_role},
          {'role':'user','content':user_prompt}
          ]

In [36]:
print(get_response(messages))

The most likely cause of this patient’s symptoms is **D: Von Willebrand disease**. 

This condition explains her menorrhagia and easy bruising, and the laboratory findings of a prolonged PTT with a normal platelet count can indicate a deficiency in von Willebrand factor, which is essential for platelet function and coagulation. The family history of similar symptoms further supports this diagnosis.


In [38]:
fifth_question_answer
# check if the answer is accurate

'Von Willebrand disease'

# One Shot Prompting

In [39]:
user_prompt = f'''Question:{first_question}
Answer:{first_question_answer}

Question:{fifth_question}
Answer:
'''

In [40]:
messages=[{'role':'system','content':system_role},
          {'role':'user','content':user_prompt}
          ]

In [41]:
print(get_response(messages))

The best answer is: D - Von Willebrand disease. 

This patient has menorrhagia and easy bruising, which are indicative of a bleeding disorder. The normal platelet count and the prolonged APTT (PTT) suggest a deficiency in factor VIII, which is commonly associated with Von Willebrand disease, especially given the patient's family history of similar bleeding issues. Hemophilia A would not typically present with a normal platelet count.


# Zero Shot Prompting

In [42]:
user_prompt = f'''Question:{fifth_question}
Answer:
'''

In [43]:
messages=[{'role':'system','content':system_role},
          {'role':'user','content':user_prompt}
          ]

In [44]:
print(get_response(messages))

The symptoms presented by the patient, including menorrhagia, easy bruising, and a prolonged PTT with a normal PT, suggest a bleeding disorder. Given the family history of similar symptoms, this points towards a hereditary condition.

Among the options provided:
- **Hemophilia A** typically presents with a very low factor VIII level leading to prolonged PTT, but it usually does not present with menorrhagia predominantly and is more common in males.
- **Lupus anticoagulant** is associated with antiphospholipid syndrome, which does not typically cause easy bruisability or menorrhagia and affects the PT and PTT differently.
- **Protein C deficiency** can lead to thrombosis rather than bleeding issues, and does not fit with the described symptoms.
- **Von Willebrand disease** is characterized by both bleeding (like menorrhagia and easy bruising) and a prolonged PTT due to the deficiency in von Willebrand factor, which is also associated with a family history.

Based on these considerations